<a href="https://colab.research.google.com/github/Anonymus373/Eco-Sentinel/blob/main/Eco_sentinel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install earthengine-api geemap rasterio geopandas scikit-learn pyod ruptures

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.8/219.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 11.1 MB/s eta 0:00:00


In [ ]:
import ee
ee.Authenticate()

True

In [ ]:
import ee
ee.Authenticate(auth_mode='notebook')


True

In [ ]:
import ee

ee.Initialize(project='ecosentinel-ml')

In [ ]:
point = ee.Geometry.Point([78.5, 27.2])

image = ee.ImageCollection("COPERNICUS/S2_SR") \
    .filterBounds(point) \
    .first()

print(image.getInfo())

/usr/local/lib/python3.12/dist-packages/ee/deprecation.py:207: DeprecationWarning: 

Attention required for COPERNICUS/S2_SR! You are using a deprecated asset.
To make sure your code keeps working, please update it.
Learn more: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR

  warnings.warn(warning, category=DeprecationWarning)


{'type': 'Image', 'bands': [{'id': 'B1', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [1830, 1830], 'crs': 'EPSG:32644', 'crs_transform': [60, 0, 199980, 0, -60, 3100020]}, {'id': 'B2', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [10980, 10980], 'crs': 'EPSG:32644', 'crs_transform': [10, 0, 199980, 0, -10, 3100020]}, {'id': 'B3', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [10980, 10980], 'crs': 'EPSG:32644', 'crs_transform': [10, 0, 199980, 0, -10, 3100020]}, {'id': 'B4', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [10980, 10980], 'crs': 'EPSG:32644', 'crs_transform': [10, 0, 199980, 0, -10, 3100020]}, {'id': 'B5', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [5490, 5490], 'crs': 'EPSG:32644', 'crs_transform': [20, 0, 199980, 0, -20, 3100020

In [ ]:
import geemap

region = ee.Geometry.Rectangle([72.5, 26.5, 78.5, 30.5]) # Aravalli region

collection = (
    ee.ImageCollection("COPERNICUS/S2_SR")
    .filterBounds(region)
    .filterDate("2020-01-01","2024-01-01")
)

def ndvi(img):
    return img.normalizedDifference(['B8','B4']).rename('NDVI')

ndvi_collection = collection.map(ndvi)

In [ ]:
import numpy as np

def compute_trend(ndvi_values):
    x = np.arange(len(ndvi_values))
    slope = np.polyfit(x, ndvi_values,1)[0]
    return slope

In [ ]:
night = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1")

night_filtered = night.filterBounds(region).filterDate("2020-01-01","2024-01-01")

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

data = pd.DataFrame({
    "ndvi_slope": np.random.normal(-0.02,0.01,100),
    "moisture_change": np.random.normal(-0.01,0.02,100),
    "nightlight_volatility": np.random.normal(0.2,0.1,100),
    "fragmentation_index": np.random.normal(0.3,0.15,100)
})

data.to_csv("features.csv", index=False)

print("features.csv created")

features.csv created


In [ ]:
from sklearn.ensemble import IsolationForest
import pandas as pd

data = pd.read_csv("features.csv")

model = IsolationForest(
    contamination=0.05,
    random_state=42
)

model.fit(data)

scores = model.decision_function(data)
anomaly = model.predict(data)

data["anomaly"] = anomaly

print(data.head())

   ndvi_slope  moisture_change  nightlight_volatility  fragmentation_index  \
0   -0.015033        -0.038307               0.235779             0.175651   
1   -0.021383        -0.018413               0.256078             0.215973   
2   -0.013523        -0.016854               0.308305             0.412094   
3   -0.004770        -0.026046               0.305380             0.391556   
4   -0.022342        -0.013226               0.062233             0.296865   

   anomaly  
0        1  
1        1  
2        1  
3        1  
4        1  


In [ ]:
data["risk_score"] = (1 - scores) * 100

In [ ]:
def classify(score):
    if score < 30:
        return "Low"
    elif score < 60:
        return "Drift"
    elif score < 85:
        return "Collapse"
    else:
        return "Immediate"

data["risk_level"] = data["risk_score"].apply(classify)

In [ ]:
data.to_json("district_risk.json", orient="records")

In [ ]:
import geemap

region = ee.Geometry.Rectangle([72.5, 26.5, 78.5, 30.5]) # Aravalli region

collection = (
    ee.ImageCollection("COPERNICUS/S2_SR")
    .filterBounds(region)
    .filterDate("2020-01-01","2024-01-01")
)

def ndvi(img):
    return img.normalizedDifference(['B8','B4']).rename('NDVI')

ndvi_collection = collection.map(ndvi)

In [ ]:
import numpy as np

def compute_trend(ndvi_values):
    x = np.arange(len(ndvi_values))
    slope = np.polyfit(x, ndvi_values,1)[0]
    return slope

In [ ]:
night = ee.ImageCollection("NOAA/VIIRS/DNB/MONTHLY_V1")

night_filtered = night.filterBounds(region).filterDate("2018-01-01","2024-01-01")

In [ ]:
from sklearn.ensemble import IsolationForest
import pandas as pd

data = pd.read_csv("features.csv")

model = IsolationForest(
    contamination=0.05,
    random_state=42
)

model.fit(data)

scores = model.decision_function(data)
anomaly = model.predict(data)

In [ ]:
from sklearn.cluster import DBSCAN

cluster = DBSCAN(eps=0.5,min_samples=5)

labels = cluster.fit_predict(data)

In [ ]:
def collapse_score(ndvi, moisture, nightlight, fragmentation):

    score = (
        0.35 * ndvi +
        0.25 * moisture +
        0.20 * nightlight +
        0.20 * fragmentation
    )

    return score

In [ ]:
data.to_json("district_risk.json")

In [ ]:
from sklearn.ensemble import IsolationForest
import pandas as pd

data = pd.read_csv("features.csv")

model = IsolationForest(
    contamination=0.05,
    random_state=42
)

model.fit(data)

scores = model.decision_function(data)
anomaly = model.predict(data)

data["anomaly"] = anomaly

print(data.head())

   ndvi_slope  moisture_change  nightlight_volatility  fragmentation_index  \
0   -0.015033        -0.038307               0.235779             0.175651   
1   -0.021383        -0.018413               0.256078             0.215973   
2   -0.013523        -0.016854               0.308305             0.412094   
3   -0.004770        -0.026046               0.305380             0.391556   
4   -0.022342        -0.013226               0.062233             0.296865   

   anomaly  
0        1  
1        1  
2        1  
3        1  
4        1  
